# Wan 2.1 T2V 1.3B on a T4: the GPU spike

Runs `worker/wan.py` (the class the worker ships) on Kaggle's free T4 and fills a pass/fail table.
Gate: decide the final `duration_seconds` list and the step count (spec section 5).

**Set up before "Run all"**
- Settings, Accelerator: **GPU T4 x2** (the notebook uses only one, like a g4dn.2xlarge).
- Settings, Internet: **On** (needs a phone-verified account). No token needed; the model is ungated.
- About 30 GB of weights download to `/tmp`; the whole run takes roughly 40 to 60 minutes.

At the end, copy the **REPORT** cell output and paste it back. It holds no secrets.
Kaggle has about 29 GB RAM; a g4dn.2xlarge has 32 GB, so a pass here is slightly conservative.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one T4, like the real instance (must be set before torch loads)
import json, shutil, subprocess, sys, time

REPORT = []
def check(name, status, detail=""):
    """status: PASS, FAIL or INFO"""
    REPORT.append({"check": name, "status": status, "detail": detail})
    print(f"[{status}] {name}: {detail}")

import psutil, torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
vram = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
check("GPU is a T4", "PASS" if "T4" in gpu else "FAIL", f"{gpu}, {vram:.1f} GiB VRAM")
check("RAM", "INFO", f"{psutil.virtual_memory().total / 2**30:.1f} GiB total (g4dn.2xlarge has 32)")
free_tmp = shutil.disk_usage("/tmp").free / 2**30
check("Disk for weights in /tmp", "PASS" if free_tmp > 40 else "FAIL", f"{free_tmp:.0f} GiB free, need about 35")
check("torch", "INFO", torch.__version__ + " (cuda " + str(torch.version.cuda) + "); the image will carry torch 2.8.0")


In [ ]:
PINS = ['diffusers==0.35.2', 'transformers==4.57.6', 'accelerate==1.15.0', 'safetensors==0.8.0', 'sentencepiece==0.2.2', 'ftfy==6.3.1', 'regex==2026.9.29', 'imageio==2.38.1', 'imageio-ffmpeg==0.6.0']
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
import importlib.metadata as md
got = {p: md.version(p) for p in ("diffusers", "transformers", "accelerate", "safetensors", "sentencepiece", "ftfy", "imageio", "imageio-ffmpeg")}
check("pinned packages installed", "PASS" if r.returncode == 0 else "FAIL", json.dumps(got))


In [ ]:
# The worker's own files, embedded by scripts/make_spike_notebook.py
import pathlib
FILES = {'__init__.py': '"""The GPU worker: takes jobs off the queue, runs the model, uploads the clip (spec section 4)."""\n', 'errors.py': '"""The ways a job attempt can end other than success."""\n\n\nclass Aborted(Exception):\n    """The generator stopped early because `should_abort()` turned true."""\n\n\nclass BadMessage(Exception):\n    """A queue message that cannot be tied to a job (no usable job_id or sub). Nobody to tell."""\n\n\nclass PermanentJobError(Exception):\n    """The job can never succeed. Becomes a `.failed` marker the user sees, so `message` must be\n    safe to show: no paths, no stack traces."""\n\n    def __init__(self, code, message, field=None):\n        super().__init__(message)\n        self.code = code\n        self.message = message\n        self.field = field\n\n    def to_json(self):\n        out = {"code": self.code}\n        if self.field:\n            out["field"] = self.field\n        out["message"] = self.message\n        return out\n', 'wan.py': '"""Wan 2.1 T2V 1.3B through diffusers\' WanPipeline (spec section 4, "Model").\n\nUNVERIFIED: nothing here has run on a GPU yet. The Kaggle spike runs this same class and checks\nspeed, memory, the step count and whether fp16 gives clean frames. Until then treat the numbers\nbelow (steps, guidance, flow_shift) as the documented defaults, not measured ones.\n\ntorch and diffusers are imported inside methods, so the rest of the worker (and its tests)\nload without them.\n"""\nimport gc\nimport pathlib\nimport random\n\nfrom .errors import Aborted\n\nWIDTH, HEIGHT, FPS = 832, 480, 16\nGUIDANCE_SCALE = 5.0\nFLOW_SHIFT = 3.0          # diffusers\' Wan docs: 3.0 for 480p, 5.0 for 720p\nFRAMES = {"3": 49, "5": 81}   # 16 fps, and the model needs 4n+1 frames\nMAX_SEED = 2147483647\n\n\ndef num_frames(duration_seconds):\n    return FRAMES[duration_seconds]\n\n\ndef pick_seed(params, rng=random):\n    seed = params.get("seed")\n    return seed if seed is not None else rng.randint(0, MAX_SEED)\n\n\nclass WanGenerator:\n    def __init__(self, models_dir, model_name, steps):\n        self.path = str(pathlib.Path(models_dir) / model_name)\n        self.steps = steps\n        self.pipe = None\n\n    def load(self):\n        """Load the weights once, at start. The T4 has no bf16: transformer and text encoder in\n        fp16, VAE in fp32 (spec section 4). Components move to the GPU only while they run."""\n        import torch\n        from diffusers import AutoencoderKLWan, UniPCMultistepScheduler, WanPipeline\n\n        vae = AutoencoderKLWan.from_pretrained(self.path, subfolder="vae", torch_dtype=torch.float32)\n        pipe = WanPipeline.from_pretrained(self.path, vae=vae, torch_dtype=torch.float16)\n        pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=FLOW_SHIFT)\n        pipe.enable_model_cpu_offload()\n        self.pipe = pipe\n\n    def generate(self, job, inputs, should_abort, out_dir):\n        error = None\n        try:\n            return self._generate(job, should_abort, out_dir)\n        except Aborted as e:\n            # The traceback holds the pipeline\'s tensors, so drop it before freeing memory.\n            error = e.with_traceback(None)\n        finally:\n            if error is None:\n                self._free()\n        self._free()\n        raise error\n\n    def _free(self):\n        """An abort skips diffusers\' own cleanup: the transformer stayed on the GPU and the next\n        job ran out of memory (Kaggle spike). Put the model back on the CPU and drop the cache."""\n        import torch\n\n        self.pipe.maybe_free_model_hooks()\n        gc.collect()\n        torch.cuda.empty_cache()\n\n    def _generate(self, job, should_abort, out_dir):\n        import torch\n        from diffusers.utils import export_to_video\n\n        if should_abort():\n            raise Aborted()\n\n        def on_step(pipe, step, timestep, callback_kwargs):\n            if should_abort():\n                raise Aborted()          # leaves the denoising loop between two steps\n            return callback_kwargs\n\n        seed = pick_seed(job.params)\n        frames = self.pipe(\n            prompt=job.prompt,\n            negative_prompt=job.params.get("negative_prompt") or None,\n            width=WIDTH, height=HEIGHT,\n            num_frames=num_frames(job.params["duration_seconds"]),\n            num_inference_steps=self.steps,\n            guidance_scale=GUIDANCE_SCALE,\n            generator=torch.Generator("cpu").manual_seed(seed),\n            callback_on_step_end=on_step,\n        ).frames[0]\n        out = pathlib.Path(out_dir) / "clip.mp4"\n        export_to_video(frames, str(out), fps=FPS)\n        return out\n'}
root = pathlib.Path("/kaggle/working/spike")
(root / "worker").mkdir(parents=True, exist_ok=True)
for name, text in FILES.items():
    (root / "worker" / name).write_text(text)
sys.path.insert(0, str(root))
from worker.wan import WanGenerator, num_frames
from worker.errors import Aborted
print(sorted(p.name for p in (root / "worker").iterdir()))


In [ ]:
from huggingface_hub import snapshot_download
WEIGHTS = "/tmp/wan-weights/wan2-1-t2v-1-3b"      # same layout as /data/models/<model> on the instance
t = time.time()
snapshot_download("Wan-AI/Wan2.1-T2V-1.3B-Diffusers", revision="0fad780a534b6463e45facd96134c9f345acfa5b", local_dir=WEIGHTS,
                  allow_patterns=["model_index.json", "scheduler/*", "text_encoder/*", "tokenizer/*", "transformer/*", "vae/*"])
size = sum(f.stat().st_size for f in pathlib.Path(WEIGHTS).rglob("*") if f.is_file()) / 2**30
check("weights downloaded", "PASS" if 25 < size < 40 else "FAIL", f"{size:.1f} GiB in {time.time() - t:.0f} s (spec says about 29)")
os.environ["HF_HUB_OFFLINE"] = "1"                 # the image never reaches out to Hugging Face
os.environ["TRANSFORMERS_OFFLINE"] = "1"


In [ ]:
gen = WanGenerator("/tmp/wan-weights", "wan2-1-t2v-1-3b", steps=30)
t = time.time()
try:
    gen.load()
    check("model loads (fp16 transformer and text encoder, fp32 VAE, cpu offload)", "PASS",
          f"{time.time() - t:.0f} s, RAM now {psutil.Process().memory_info().rss / 2**30:.1f} GiB")
except Exception as e:
    check("model loads", "FAIL", repr(e)[:400])
    raise


In [ ]:
import types
import numpy as np, imageio
from PIL import Image
from IPython.display import display

def job(prompt, seconds, seed):
    return types.SimpleNamespace(prompt=prompt, params={"seed": seed, "duration_seconds": seconds, "negative_prompt": ""})

def read_frames(path):
    rd = imageio.get_reader(str(path), "ffmpeg")
    frames = [np.asarray(f) for f in rd]
    rd.close()
    return frames

def contact_sheet(frames, name):
    idx = np.linspace(0, len(frames) - 1, 6).astype(int)
    thumbs = [Image.fromarray(frames[i]).resize((416, 240)) for i in idx]
    sheet = Image.new("RGB", (416 * 3, 240 * 2))
    for n, im in enumerate(thumbs):
        sheet.paste(im, ((n % 3) * 416, (n // 3) * 240))
    sheet.save(f"/kaggle/working/{name}.png")
    display(sheet)

def quality(frames):
    a = np.stack(frames).astype(np.float32)
    motion = float(np.abs(np.diff(a, axis=0)).mean())
    return {"mean": round(float(a.mean()), 1), "std": round(float(a.std()), 1), "motion": round(motion, 2),
            "black_frames": int(sum(f.max() < 8 for f in frames))}

def run(label, prompt, seconds, steps, seed, should_abort=lambda: False, show=True):
    """One generation. Returns (frames, seconds, peak_vram_gib) or raises."""
    gen.steps = steps
    out = pathlib.Path(f"/tmp/out-{label}"); out.mkdir(exist_ok=True)
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    path = gen.generate(job(prompt, seconds, seed), {}, should_abort, out)
    secs = time.time() - t
    peak = torch.cuda.max_memory_allocated() / 2**30
    frames = read_frames(path)
    if show:
        contact_sheet(frames, label)
    return frames, secs, peak

def judge(label, frames, secs, peak, want_frames, limit_s=900):
    q = quality(frames)
    clean = q["black_frames"] == 0 and q["std"] > 10 and q["mean"] > 8 and not np.isnan(q["mean"])
    moving = q["motion"] > 0.5
    ok = len(frames) == want_frames and clean and moving and secs <= limit_s
    why = f"{len(frames)} frames (want {want_frames}), {secs:.0f} s, peak VRAM {peak:.1f} GiB (contact sheet attached), quality {q}"
    if not clean: why += " | BLACK/FLAT FRAMES: try fp32 transformer (spec section 5)"
    if not moving: why += " | FRAMES BARELY MOVE"
    if secs > limit_s: why += f" | over {limit_s} s"
    check(label, "PASS" if ok else "FAIL", why)


## 3 s clip (49 frames)

In [ ]:
PROMPT_A = "A cat walking through a sunlit garden, cinematic, detailed fur"
try:
    f3, s3, p3 = run("clip-3s-30steps", PROMPT_A, "3", 30, 1234)
    judge("3 s clip (49 frames, 30 steps)", f3, s3, p3, 49)
except Exception as e:
    check("3 s clip (49 frames, 30 steps)", "FAIL", repr(e)[:400])


## 5 s clip (81 frames)

In [ ]:
PROMPT_B = "A red balloon floating up above a city street at sunset, soft light"
try:
    f5, s5, p5 = run("clip-5s-30steps", PROMPT_B, "5", 30, 99)
    judge("5 s clip (81 frames, 30 steps)", f5, s5, p5, 81)
except Exception as e:
    check("5 s clip (81 frames, 30 steps)", "FAIL", repr(e)[:400] + " (if out of memory: drop the 5 s option)")


## Step count: 20 vs 30

In [ ]:
# Is 20 steps good enough? Same seed, same prompt as the 3 s clip: compare the two contact sheets by eye.
try:
    f3b, s3b, p3b = run("clip-3s-20steps", PROMPT_A, "3", 20, 1234)
    check("3 s clip at 20 steps", "INFO", f"{s3b:.0f} s vs {s3:.0f} s at 30 steps; look at clip-3s-20steps.png against clip-3s-30steps.png")
except Exception as e:
    check("3 s clip at 20 steps", "FAIL", repr(e)[:300])


## Abort between steps

In [ ]:
# should_abort flips true 20 s into a run; the generator must raise Aborted at the next step.
flip = {}
def abort_after_20s(_t0=[None]):
    if _t0[0] is None: _t0[0] = time.time()
    if time.time() - _t0[0] > 20:
        flip.setdefault("at", time.time())
        return True
    return False
try:
    run("abort", PROMPT_A, "3", 30, 5, should_abort=abort_after_20s, show=False)
    check("abort between steps", "FAIL", "the run finished instead of stopping")
except Aborted:
    lag = time.time() - flip["at"]
    freed = torch.cuda.memory_allocated() / 2**30
    check("abort between steps", "PASS" if lag < 60 else "FAIL", f"stopped {lag:.1f} s after the flag (one step is the limit); GPU memory still held {freed:.1f} GiB")
except Exception as e:
    check("abort between steps", "FAIL", repr(e)[:300])


## Seeds

In [ ]:
# Same seed twice should repeat (a hint, not a guarantee on GPUs); another seed must differ. 8 steps keeps it quick.
try:
    a, *_ = run("seed-a1", PROMPT_A, "3", 8, 7, show=False)
    b, *_ = run("seed-a2", PROMPT_A, "3", 8, 7, show=False)
    c, *_ = run("seed-b", PROMPT_A, "3", 8, 8, show=False)
    same = float(np.abs(np.stack(a).astype(float) - np.stack(b).astype(float)).mean())
    diff = float(np.abs(np.stack(a).astype(float) - np.stack(c).astype(float)).mean())
    check("same seed repeats", "INFO", f"mean pixel difference {same:.2f} (0 = identical)")
    check("different seed differs", "PASS" if diff > 2 * max(same, 0.5) else "FAIL", f"mean pixel difference {diff:.2f}")
except Exception as e:
    check("seed behaviour", "FAIL", repr(e)[:300])


## Report

In [ ]:
# REPORT: copy everything this cell prints and paste it back.
verdict = "ALL PASS" if all(r["status"] != "FAIL" for r in REPORT) else "HAS FAILURES"
print(f"=== SPIKE REPORT: {verdict} ===")
for r in REPORT:
    print(f"{r['status']:5} | {r['check']} | {r['detail']}")
json.dump(REPORT, open("/kaggle/working/spike_report.json", "w"), indent=1)
print("\nAlso saved: spike_report.json and the contact-sheet PNGs in /kaggle/working (Output tab).")
print("Questions this answers: loads on a T4 | seconds per 3 s and 5 s clip | black or distorted frames | VAE fp32 | 20 vs 30 steps | abort works")
